<a href="https://colab.research.google.com/github/florianaewing/AD325_Exercises/blob/main/OptimumDeparture_2014_2024_FE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Florian Ewing, Margaret Keau, John Farnandez, Lauren Connely
# Professor Ix Procopios
# Software Design and Implementation
# Adapted: 2014–2024 BTS Flight Delay Dataset

# Optimum Departure Location & Airline by Region Predictor
# (2014–2024 BTS Dataset Adaptation)

This notebook adapts the original experiment to the larger **2014–2024 Delayed Flight Data**
dataset from Kaggle (`JOHNFF09/2014-2024-delayed-flight-data`).

The dataset is a raw BTS export. Every categorical field is stored as a numeric code
that must be decoded using official BTS lookup tables before results are human-readable.

### Lookup tables fetched from BTS
| Column | Lookup | Gives us |
|---|---|---|
| `ORIGIN_AIRPORT_ID` | `L_AIRPORT_ID` | City name + state |
| `ORIGIN_CITY_MARKET_ID` | `L_CITY_MARKET_ID` | City name + state (backup) |
| `OP_UNIQUE_CARRIER` | `L_UNIQUE_CARRIERS` | Full airline name |

### Pipeline overview
- **Cell 1** — Kaggle auth + dataset download
- **Cell 2** — Fetch all three BTS lookup tables
- **Cell 3** — Load flight CSV, rename columns, decode all coded fields
- **Cell 4** — Create delay target + class weights
- **Cell 5** — Region mapping (state → region)
- **Cell 6** — Feature engineering
- **Cell 7** — Train Random Forest
- **Cell 8** — Recommender function (outputs human-readable names)
- **Cell 9** — User interaction

# Cell 1 — Kaggle Auth & Download

In [4]:
# ===== Cell 1: Install & Download =====
import os
import pandas as pd
import numpy as np
import kagglehub

dataset_path = kagglehub.dataset_download("JOHNFF09/2014-2024-delayed-flight-data")
print("Downloaded to:", dataset_path)

dataset_path = dataset_path  # path to the directory with CSV files

print("\nFiles downloaded:")
for f in sorted(os.listdir(dataset_path)):
    size_mb = os.path.getsize(os.path.join(dataset_path, f)) / 1e6
    print(f"  {f:50s}  {size_mb:7.1f} MB")

100%|██████████| 975M/975M [00:06<00:00, 154MB/s]

Extracting files...


Downloaded to: /root/.cache/kagglehub/datasets/JOHNFF09/2014-2024-delayed-flight-data/versions/1

Files downloaded:
  flights10year.csv                                    6002.9 MB


# Cell 2 — Fetch BTS Lookup Tables

Fetches three lookup tables directly from the BTS transtats server to decode
the numeric codes in the dataset into human-readable names.

- `L_AIRPORT_ID` → decodes `ORIGIN_AIRPORT_ID` → city + state
- `L_CITY_MARKET_ID` → decodes `ORIGIN_CITY_MARKET_ID` → city + state (backup)
- `L_UNIQUE_CARRIERS` → decodes `OP_UNIQUE_CARRIER` → full airline name

In [8]:
# ===== Cell 2: Fetch BTS Lookup Tables =====
import pandas as pd
import numpy as np
import requests, io, zipfile

BTS_LOOKUP_URLS = {
    "airport":     "https://www.transtats.bts.gov/Download_Lookup.asp?Y11x72=Y_NVecbeg_VQ",
    "city_market": "https://www.transtats.bts.gov/Download_Lookup.asp?Y11x72=Y_PVgl_ZNeXRg_VQ",
    "carrier":     "https://www.transtats.bts.gov/Download_Lookup.asp?Y11x72=Y_haVdhR_PNeeVRef",
}

def fetch_bts_lookup(url):
    """Fetch a BTS lookup table. Handles both plain CSV and zip responses."""
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    raw = resp.content
    if raw[:2] == b"PK":
        with zipfile.ZipFile(io.BytesIO(raw)) as z:
            csv_name = [n for n in z.namelist() if n.endswith(".csv")][0]
            with z.open(csv_name) as f:
                return pd.read_csv(f)
    else:
        return pd.read_csv(io.StringIO(raw.decode("utf-8", errors="replace")))

# ---- Airport ID lookup ----
# Format: "City, ST: Airport Name"
raw_airport = fetch_bts_lookup(BTS_LOOKUP_URLS["airport"])
raw_airport.columns = ["Code", "Description"]
raw_airport["Code"] = pd.to_numeric(raw_airport["Code"], errors="coerce")
raw_airport = raw_airport.dropna(subset=["Code"])
raw_airport["Code"] = raw_airport["Code"].astype(int)
raw_airport["State"]     = raw_airport["Description"].str.extract(r",\s*([A-Z]{2}):")
raw_airport["CityState"] = raw_airport["Description"].str.extract(r"^(.+):")
airport_lookup = raw_airport[["Code", "State", "CityState"]].dropna(subset=["State"])
print(f"Airport lookup: {len(airport_lookup)} entries — sample:")
print(airport_lookup.head(5).to_string(index=False))

# ---- City Market lookup ----
# Format: "City, ST"
raw_city = fetch_bts_lookup(BTS_LOOKUP_URLS["city_market"])
raw_city.columns = ["Code", "Description"]
raw_city["Code"] = pd.to_numeric(raw_city["Code"], errors="coerce")
raw_city = raw_city.dropna(subset=["Code"])
raw_city["Code"] = raw_city["Code"].astype(int)
raw_city["State"]    = raw_city["Description"].str.extract(r",\s*([A-Z]{2})$")
raw_city["CityName"] = raw_city["Description"].str.extract(r"^(.+),\s*[A-Z]{2}$")
city_lookup = raw_city[["Code", "CityName", "State"]].dropna(subset=["State"])
print(f"\nCity market lookup: {len(city_lookup)} entries — sample:")
print(city_lookup.head(5).to_string(index=False))

# ---- Carrier lookup ----
raw_carrier = fetch_bts_lookup(BTS_LOOKUP_URLS["carrier"])
raw_carrier.columns = ["Code", "Description"]
raw_carrier = raw_carrier.dropna()
raw_carrier["Code"]        = raw_carrier["Code"].astype(str).str.strip()
raw_carrier["AirlineName"] = raw_carrier["Description"].str.strip()
carrier_lookup = raw_carrier[["Code", "AirlineName"]]
print(f"\nCarrier lookup: {len(carrier_lookup)} entries — sample:")
print(carrier_lookup.head(5).to_string(index=False))

Airport lookup: 2961 entries — sample:
 Code State            CityState
10001    AK     Afognak Lake, AK
10003    AK Granite Mountain, AK
10004    AK              Lik, AK
10005    AK     Little Squaw, AK
10006    AK         Kizhuyak, AK

City market lookup: 2491 entries — sample:
 Code         CityName State
30001     Afognak Lake    AK
30003 Granite Mountain    AK
30004              Lik    AK
30005     Little Squaw    AK
30006         Kizhuyak    AK

Carrier lookup: 1774 entries — sample:
Code                                          AirlineName
 02Q                                        Titan Airways
 05Q                                  Comlux Aviation, AG
 06Q                        Master Top Linhas Aereas Ltd.
 07Q                                  Flair Airlines Ltd.
 09Q Swift Air, LLC d/b/a Eastern Air Lines d/b/a Eastern


# Cell 3 — Load Flight Data & Decode All Coded Fields

Loads the raw CSV, renames BTS columns to internal names, then immediately
joins all three lookup tables to add human-readable fields.
Both the numeric IDs (used by the model) and the readable names
(used in output) are kept in the dataframe.

In [9]:
# ===== Cell 3: Load Flight Data & Decode =====

KEEP_RAW = [
    "FL_DATE",
    "OP_UNIQUE_CARRIER",
    "ORIGIN_AIRPORT_ID",
    "ORIGIN_CITY_MARKET_ID",
    "DEST_AIRPORT_ID",
    "DEST_CITY_MARKET_ID",
    "CRS_DEP_TIME",
    "DEP_TIME",
    "DEP_DELAY_NEW",
    "ARR_DELAY_NEW",
    "DISTANCE",
    "CARRIER_DELAY",
    "WEATHER_DELAY",
    "NAS_DELAY",
    "SECURITY_DELAY",
    "LATE_AIRCRAFT_DELAY",
]

BTS_RENAME = {
    "FL_DATE":               "FlightDate",
    "OP_UNIQUE_CARRIER":     "Airline_Code",
    "ORIGIN_AIRPORT_ID":     "Dep_Airport_ID",
    "ORIGIN_CITY_MARKET_ID": "Dep_Market_ID",
    "DEST_AIRPORT_ID":       "Arr_Airport_ID",
    "DEST_CITY_MARKET_ID":   "Arr_Market_ID",
    "CRS_DEP_TIME":          "Sched_Dep_Time",
    "DEP_TIME":              "Dep_Time",
    "DEP_DELAY_NEW":         "Dep_Delay",
    "ARR_DELAY_NEW":         "Arr_Delay",
    "DISTANCE":              "Distance",
    "CARRIER_DELAY":         "Carrier_Delay",
    "WEATHER_DELAY":         "Weather_Delay",
    "NAS_DELAY":             "NAS_Delay",
    "SECURITY_DELAY":        "Security_Delay",
    "LATE_AIRCRAFT_DELAY":   "LateAircraft_Delay",
}

csv_files = sorted([
    os.path.join(dataset_path, f)
    for f in os.listdir(dataset_path)
    if f.endswith(".csv")
])

print(f"Found {len(csv_files)} CSV file(s). Loading...")
chunks = []
for fp in csv_files:
    header = pd.read_csv(fp, nrows=0).columns.tolist()
    cols_to_read = [c for c in KEEP_RAW if c in header]
    df = pd.read_csv(fp, usecols=cols_to_read, low_memory=False)
    df.rename(columns=BTS_RENAME, inplace=True)
    df["FlightDate"] = pd.to_datetime(df["FlightDate"], format="mixed")
    chunks.append(df)
    print(f"  Loaded {fp.split('/')[-1]:50s}  rows={len(df):,}")

flights = pd.concat(chunks, ignore_index=True)
print(f"\nTotal rows loaded: {len(flights):,}")

# ---- Decode airport IDs ----
flights = flights.merge(
    airport_lookup.rename(columns={"Code": "Dep_Airport_ID",
                                    "State": "Dep_State",
                                    "CityState": "Dep_CityState"}),
    on="Dep_Airport_ID", how="left"
)
flights = flights.merge(
    airport_lookup.rename(columns={"Code": "Arr_Airport_ID",
                                    "State": "Arr_State",
                                    "CityState": "Arr_CityState"}),
    on="Arr_Airport_ID", how="left"
)

# ---- Decode city market IDs (backup state source) ----
flights = flights.merge(
    city_lookup.rename(columns={"Code": "Dep_Market_ID",
                                 "CityName": "Dep_CityName",
                                 "State": "Dep_Market_State"}),
    on="Dep_Market_ID", how="left"
)
# Use airport state first, fall back to city market state
flights["Dep_State"] = flights["Dep_State"].fillna(flights["Dep_Market_State"])

# ---- Decode carrier codes ----
flights = flights.merge(
    carrier_lookup.rename(columns={"Code": "Airline_Code",
                                    "AirlineName": "Airline_Name"}),
    on="Airline_Code", how="left"
)

print(f"\nDecoded flights shape: {flights.shape}")
print("\nSample decoded rows:")
print(flights[[
    "FlightDate", "Airline_Code", "Airline_Name",
    "Dep_Airport_ID", "Dep_CityState", "Dep_State", "Dep_Delay"
]].dropna(subset=["Airline_Name", "Dep_CityState"]).head(5).to_string(index=False))

Found 1 CSV file(s). Loading...
  Loaded flights10year.csv                                   rows=63,626,717

Total rows loaded: 63,626,717

Decoded flights shape: (63626717, 23)

Sample decoded rows:
FlightDate Airline_Code      Airline_Name  Dep_Airport_ID                  Dep_CityState Dep_State  Dep_Delay
2024-01-01           9E Endeavor Air Inc.           10135 Allentown/Bethlehem/Easton, PA        PA        0.0
2024-01-01           9E Endeavor Air Inc.           10135 Allentown/Bethlehem/Easton, PA        PA        9.0
2024-01-01           9E Endeavor Air Inc.           10185                 Alexandria, LA        LA        0.0
2024-01-01           9E Endeavor Air Inc.           10185                 Alexandria, LA        LA        0.0
2024-01-01           9E Endeavor Air Inc.           10208                    Augusta, GA        GA        0.0


# Cell 4 — Create Target Variables & Class Weights

In [11]:
# ===== Cell 4: Target Variables =====

flights = flights.dropna(subset=["Dep_Delay"])

# Binary target: 0 = on time / < 15 min,  1 = delayed >= 15 min
flights["Delay_Class"] = np.where(flights["Dep_Delay"] >= 15, 1, 0)

delay_counts = flights["Delay_Class"].value_counts()
delay_pct    = flights["Delay_Class"].value_counts(normalize=True) * 100

print("Delay Class Counts:");  print(delay_counts)
print("\nDelay Class %:");      print(delay_pct.round(2))

total    = len(flights)
weight_0 = total / (2 * delay_counts[0])
weight_1 = total / (2 * delay_counts[1])
class_weights = {0: weight_0, 1: weight_1}
print("\nSuggested class weights:", class_weights)

Delay Class Counts:
Delay_Class
0    50927248
1    11470970
Name: count, dtype: int64

Delay Class %:
Delay_Class
0    81.62
1    18.38
Name: proportion, dtype: float64

Suggested class weights: {0: np.float64(0.6126211453640692), 1: np.float64(2.719831801495427)}


# Cell 5 — Region Mapping

Now that `Dep_State` is decoded from the BTS lookup tables, the region
mapping is identical to the original notebook.

In [12]:
# ===== Cell 5: Region Mapping =====

region_to_states = {
    "Northeast": ["ME","NH","VT","MA","RI","CT","NY","NJ","PA","MD","DE"],
    "Southeast": ["VA","WV","KY","TN","NC","SC","GA","FL","AL","MS","AR","LA"],
    "Midwest":   ["OH","IN","IL","MI","WI","MN","IA","MO","ND","SD","NE","KS"],
    "Southwest": ["TX","OK","NM","AZ"],
    "West":      ["CO","WY","MT","ID","UT","NV","CA","OR","WA","AK","HI"],
}
state_to_region = {
    state: region
    for region, states in region_to_states.items()
    for state in states
}

flights["Region"] = flights["Dep_State"].map(state_to_region)

print("Flights per region:")
print(flights["Region"].value_counts())
print(f"\nUnmapped (territories etc.): {flights['Region'].isna().sum():,}")

Flights per region:
Region
West         17065278
Southeast    15092732
Midwest      10804040
Southwest     9148306
Northeast     8369932
Name: count, dtype: int64

Unmapped (territories etc.): 1,917,930


# Cell 6 — Feature Engineering

New features vs. the original notebook:
- `Year` — 10-year span makes year a meaningful signal
- `Dep_Hour` — strong delay predictor available from BTS `CRS_DEP_TIME`
- `Weather_Delay`, `Carrier_Delay`, `NAS_Delay`, `LateAircraft_Delay` — delay
  cause columns present in this dataset; filled with 0 for on-time flights
  (accurate: no delay means no delay minutes to attribute)

In [13]:
# ===== Cell 6: Feature Engineering =====
from sklearn.preprocessing import LabelEncoder

# ---- Holiday proximity flag (2014–2024) ----
holiday_dates = []
for year in range(2014, 2025):
    holiday_dates += [f"{year}-01-01", f"{year}-07-04", f"{year}-12-25"]

approx_floating = [
    "2014-01-20","2014-02-17","2014-05-26","2014-09-01","2014-11-27",
    "2015-01-19","2015-02-16","2015-05-25","2015-09-07","2015-11-26",
    "2016-01-18","2016-02-15","2016-05-30","2016-09-05","2016-11-24",
    "2017-01-16","2017-02-20","2017-05-29","2017-09-04","2017-11-23",
    "2018-01-15","2018-02-19","2018-05-28","2018-09-03","2018-11-22",
    "2019-01-21","2019-02-18","2019-05-27","2019-09-02","2019-11-28",
    "2020-01-20","2020-02-17","2020-05-25","2020-09-07","2020-11-26",
    "2021-01-18","2021-02-15","2021-05-31","2021-09-06","2021-11-25",
    "2022-01-17","2022-02-21","2022-05-30","2022-09-05","2022-11-24",
    "2023-01-16","2023-02-20","2023-05-29","2023-09-04","2023-11-23",
    "2024-01-15","2024-02-19","2024-05-27","2024-09-02","2024-11-28",
]
us_holidays = pd.to_datetime(holiday_dates + approx_floating)

def is_near_holiday(date_series, holidays, window=2):
    flags = np.zeros(len(date_series), dtype=int)
    for h in holidays:
        diff = (date_series - h).dt.days.abs()
        flags |= (diff <= window).astype(int).values
    return flags

# ---- Temporal features ----
flights["Year"]      = flights["FlightDate"].dt.year
flights["Month"]     = flights["FlightDate"].dt.month
flights["DayOfWeek"] = flights["FlightDate"].dt.dayofweek
flights["IsWeekend"] = (flights["DayOfWeek"] >= 5).astype(int)
flights["IsHoliday"] = is_near_holiday(flights["FlightDate"], us_holidays)
flights["Dep_Hour"]  = (flights["Sched_Dep_Time"] // 100).clip(0, 23)

# ---- Fill delay cause columns with 0 for on-time flights ----
for col in ["Weather_Delay", "Carrier_Delay", "NAS_Delay", "LateAircraft_Delay"]:
    flights[col] = flights[col].fillna(0)

# ---- Build model dataframe ----
FEATURE_COLS = [
    "Airline_Code",
    "Dep_Airport_ID",
    "Year",
    "Month",
    "DayOfWeek",
    "IsWeekend",
    "IsHoliday",
    "Dep_Hour",
    "Weather_Delay",
    "Carrier_Delay",
    "NAS_Delay",
    "LateAircraft_Delay",
]

model_df = flights[
    FEATURE_COLS + ["Delay_Class", "Region", "Airline_Name", "Dep_CityState"]
].dropna(subset=FEATURE_COLS + ["Delay_Class", "Region"])

# ---- Encode categoricals ----
le_airline = LabelEncoder()
le_airport = LabelEncoder()

model_df = model_df.copy()
model_df["Airline_enc"]     = le_airline.fit_transform(model_df["Airline_Code"])
model_df["Dep_Airport_enc"] = le_airport.fit_transform(model_df["Dep_Airport_ID"])

FINAL_FEATURES = [
    "Airline_enc",
    "Dep_Airport_enc",
    "Year",
    "Month",
    "DayOfWeek",
    "IsWeekend",
    "IsHoliday",
    "Dep_Hour",
    "Weather_Delay",
    "Carrier_Delay",
    "NAS_Delay",
    "LateAircraft_Delay",
]

X = model_df[FINAL_FEATURES]
y = model_df["Delay_Class"]

print(f"Feature matrix shape: {X.shape}")
print(f"Class distribution:\n{y.value_counts(normalize=True).round(3)}")

Feature matrix shape: (60480288, 12)
Class distribution:
Delay_Class
0    0.816
1    0.184
Name: proportion, dtype: float64


# Cell 7 — Train Random Forest

Samples 3 million rows for training. Class weights computed dynamically
from the actual sample distribution.

In [14]:
# ===== Cell 7: Train Random Forest =====
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

SAMPLE_SIZE = 3_000_000

sample_df = model_df.sample(n=min(SAMPLE_SIZE, len(model_df)), random_state=42)
X_s = sample_df[FINAL_FEATURES]
y_s = sample_df["Delay_Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X_s, y_s, test_size=0.2, random_state=42, stratify=y_s
)

n0 = (y_s == 0).sum()
n1 = (y_s == 1).sum()
dynamic_weights = {0: len(y_s) / (2 * n0), 1: len(y_s) / (2 * n1)}
print("Dynamic class weights:", dynamic_weights)

rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=20,
    min_samples_leaf=50,
    class_weight=dynamic_weights,
    n_jobs=-1,
    random_state=42,
)

rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
print(classification_report(y_test, y_pred, target_names=["On Time", "Delayed"]))

Dynamic class weights: {0: np.float64(0.6125104024683352), 1: np.float64(2.7220167603645327)}
              precision    recall  f1-score   support

     On Time       0.95      0.98      0.97    489788
     Delayed       0.92      0.78      0.84    110212

    accuracy                           0.95    600000
   macro avg       0.93      0.88      0.91    600000
weighted avg       0.95      0.95      0.95    600000



# Cell 8 — Recommender Function

Returns the top 5 airport+airline combos with the lowest delay probability
for a given region and date. Output uses fully decoded human-readable names:
- Airport shown as `City, ST` (e.g. `Seattle, WA`)
- Airline shown as full name (e.g. `Alaska Airlines Inc.`)

The delay cause columns are set to 0 when predicting future flights since
they are unknown in advance — this represents a baseline no-known-delay scenario.

In [15]:
# ===== Cell 8: Recommender Function =====

def get_top5_recommendations(region, date_str, dep_hour=None):
    """
    Given a region name, a date string (YYYY-MM-DD), and an optional
    departure hour (0-23), return the top 5 airport+airline combos
    with the lowest predicted delay probability, with human-readable names.
    """
    date = pd.to_datetime(date_str)

    year       = date.year
    month      = date.month
    dow        = date.dayofweek
    is_weekend = int(dow >= 5)
    is_holiday = int(any(abs((date - h).days) <= 2 for h in us_holidays))

    # All airport+airline combos that operated in this region
    combos = (
        model_df[model_df["Region"] == region]
        [["Dep_Airport_ID", "Airline_Code", "Airline_Name", "Dep_CityState"]]
        .drop_duplicates(subset=["Dep_Airport_ID", "Airline_Code"])
        .reset_index(drop=True)
    )

    if combos.empty:
        print(f"No flight combinations found for region: {region}")
        return None

    # Median departure hour per airport
    airport_medians = (
        model_df[model_df["Region"] == region]
        .groupby("Dep_Airport_ID")[["Dep_Hour"]]
        .median()
        .reset_index()
    )
    combos = combos.merge(airport_medians, on="Dep_Airport_ID", how="left")

    if dep_hour is not None:
        combos["Dep_Hour"] = int(dep_hour)

    combos["Year"]      = year
    combos["Month"]     = month
    combos["DayOfWeek"] = dow
    combos["IsWeekend"] = is_weekend
    combos["IsHoliday"] = is_holiday

    # Delay cause columns = 0 (unknown for future flights)
    combos["Weather_Delay"]      = 0
    combos["Carrier_Delay"]      = 0
    combos["NAS_Delay"]          = 0
    combos["LateAircraft_Delay"] = 0

    # Encode (handle unseen labels gracefully)
    def safe_encode(encoder, values):
        known = set(encoder.classes_)
        return np.array([
            encoder.transform([v])[0] if v in known else -1
            for v in values
        ])

    combos["Airline_enc"]     = safe_encode(le_airline, combos["Airline_Code"])
    combos["Dep_Airport_enc"] = safe_encode(le_airport, combos["Dep_Airport_ID"])

    combos = combos[
        (combos["Airline_enc"]     != -1) &
        (combos["Dep_Airport_enc"] != -1)
    ]

    X_pred = combos[FINAL_FEATURES].values
    delay_probs = rf.predict_proba(X_pred)[:, 1]

    combos["Delay_Probability"]  = delay_probs
    combos["OnTime_Probability"] = 1 - delay_probs

    top5 = (
        combos[[
            "Dep_CityState",
            "Airline_Name",
            "Delay_Probability",
            "OnTime_Probability"
        ]]
        .sort_values("Delay_Probability")
        .head(5)
        .reset_index(drop=True)
    )
    top5.index += 1
    top5["Delay_Probability"]  = top5["Delay_Probability"].map("{:.1%}".format)
    top5["OnTime_Probability"] = top5["OnTime_Probability"].map("{:.1%}".format)
    top5.columns = ["Airport (City, State)", "Airline",
                    "Delay Probability", "On-Time Probability"]
    return top5

# Cell 9 — User Interaction

In [16]:
# ===== Cell 9: User Interaction =====

VALID_REGIONS = list(region_to_states.keys())

print("=== Flight Delay Predictor (2014-2024 dataset) ===")
print(f"Available regions: {', '.join(VALID_REGIONS)}\n")

region_input = input("Enter your departure region: ").strip().title()
date_input   = input("Enter your departure date (YYYY-MM-DD): ").strip()
hour_raw     = input("Enter preferred departure hour 0-23 (or press Enter for median): ").strip()
hour_input   = int(hour_raw) if hour_raw.isdigit() else None

if region_input not in VALID_REGIONS:
    print(f"Invalid region. Please choose from: {', '.join(VALID_REGIONS)}")
else:
    try:
        results = get_top5_recommendations(region_input, date_input, dep_hour=hour_input)
        if results is not None:
            hour_str = f" at hour {hour_input}" if hour_input is not None else ""
            print(f"\nTop 5 recommendations for the {region_input} on {date_input}{hour_str}:\n")
            print(results.to_string())
    except Exception as e:
        print(f"Something went wrong: {e}")

=== Flight Delay Predictor (2014-2024 dataset) ===
Available regions: Northeast, Southeast, Midwest, Southwest, West

Enter your departure region: West
Enter your departure date (YYYY-MM-DD): 2026-03-17
Enter preferred departure hour 0-23 (or press Enter for median): 8

Top 5 recommendations for the West on 2026-03-17 at hour 8:

  Airport (City, State)           Airline Delay Probability On-Time Probability
1    Montrose/Delta, CO  Republic Airline              6.9%               93.1%
2           Durango, CO  Republic Airline              7.0%               93.0%
3         Kalispell, MT  Republic Airline              7.1%               92.9%
4        Sacramento, CA  Republic Airline              7.2%               92.8%
5           Spokane, WA  Republic Airline              7.2%               92.8%


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
